# Aula 05 - RAG Agente


## Configuração

Este notebook demonstra o padrão Agentic RAG (Geração Aumentada por Recuperação) usando o Microsoft Agent Framework.

**Pré-requisitos:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — o seu endpoint do serviço Azure AI Search
- `AZURE_SEARCH_API_KEY` — a sua chave de API do Azure AI Search
- Implementação do Azure OpenAI configurada através de variáveis de ambiente
- Azure CLI autenticado (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é Agentic RAG?

O RAG tradicional segue um pipeline fixo: recuperar documentos e depois gerar uma resposta. O **Agentic RAG** vai mais além, dando ao agente autonomia para decidir **quando** e **como** recuperar informação.

Com o Agentic RAG, o agente pode:
- **Decidir** se é necessário fazer uma recuperação antes de responder a uma pergunta
- **Escolher** qual fonte de dados ou ferramenta consultar
- **Avaliar** os resultados recuperados e realizar recuperações adicionais se a primeira tentativa for insuficiente
- **Combinar** informação de múltiplas etapas de recuperação numa resposta coerente

Isto torna o agente mais flexível e preciso em comparação com um pipeline estático de recuperar e depois gerar.


## Criar uma Ferramenta de Pesquisa

No Agentic RAG, as fontes de dados externas são encapsuladas como **ferramentas** que o agente pode invocar a pedido. Isto permite que o agente trate a obtenção de informação como mais uma ação que pode executar, em vez de um passo obrigatório.

Em baixo definimos uma base de conhecimento de viagens e expomos-na como uma ferramenta que o agente pode usar para consultar informações sobre destinos.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Construir o Agente RAG

Agora criamos um agente que é instruído a **sempre recuperar informação antes de responder**. O agente utiliza a ferramenta `search_travel_knowledge` para fundamentar as suas respostas na base de conhecimento em vez de depender dos seus próprios dados de treino.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Recuperação Iterativa — O Padrão Maker-Checker

Uma vantagem fundamental do Agentic RAG é a **recuperação iterativa**. O agente pode realizar várias rondas de pesquisa para verificar, refinar ou expandir os seus achados iniciais — semelhante a um fluxo de trabalho "maker-checker":

1. **Passo do maker**: O agente recupera informação inicial e elabora uma resposta.
2. **Passo do checker**: O agente realiza recuperações adicionais para verificar detalhes ou preencher lacunas.

Abaixo, é feita ao agente uma pergunta que exige comparar vários destinos, levando-o a pesquisar várias vezes.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Resumo

Nesta lição aprendeu a construir um sistema **Agentic RAG** usando o Microsoft Agent Framework:

- **Agentic RAG** permite que os agentes decidam de forma autónoma quando recuperar informação, tornando a recuperação dinâmica em vez de fixa.
- **Ferramentas como fontes de dados**: Bases de conhecimento externas (como o Azure AI Search) são encapsuladas como ferramentas que o agente pode invocar.
- **Recuperação iterativa**: O padrão maker-checker permite que o agente execute várias rondas de recuperação — pesquisar, verificar e refinar — antes de produzir uma resposta final.

Em produção, substituiria a base de conhecimento `TRAVEL_KNOWLEDGE_BASE` em memória por um índice real do Azure AI Search para gerir a recuperação de documentos de viagem em larga escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, por favor tenha em conta que traduções automáticas podem conter erros ou imprecisões. O documento original, no seu idioma nativo, deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se a tradução profissional por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
